In [33]:
# !pip install catboost lightgbm xgboost

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder, FunctionTransformer
from sklearn.model_selection import GroupKFold

from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
import xgboost as xgb

df = pd.read_csv('rps_data.csv', sep=',')
df['next_opp_move'] = df.groupby('match_id')['opp_move'].shift(-1)
df = df[df['next_opp_move'].notna()]

In [28]:
df.info()

<class 'pandas.DataFrame'>
Index: 1553 entries, 0 to 1841
Data columns (total 34 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           1553 non-null   int64  
 1   round              1553 non-null   int64  
 2   player_name        1553 non-null   str    
 3   win_category       1553 non-null   str    
 4   opp_match_wins     1553 non-null   int64  
 5   opp_match_winrate  1553 non-null   float64
 6   stake              1553 non-null   int64  
 7   opp_move           1553 non-null   str    
 8   my_move            1553 non-null   str    
 9   outcome            1553 non-null   str    
 10  score_me_before    1553 non-null   int64  
 11  score_opp_before   1553 non-null   int64  
 12  score_diff         1553 non-null   int64  
 13  streak_draws       1553 non-null   int64  
 14  is_last_round      1553 non-null   int64  
 15  prev1_opp_move     1553 non-null   str    
 16  prev1_my_move      1553 non-null   str  

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0. Загрузка и константы
# ------------------------------------------------------------
df_raw = pd.read_csv('rps_data.csv')
moves = ['К', 'Н', 'Б']
move_to_idx = {'К': 0, 'Н': 1, 'Б': 2}
idx_to_move = {0: 'К', 1: 'Н', 2: 'Б'}
beats = {'К': 'Н', 'Н': 'Б', 'Б': 'К'}
optimal_map = {'К': 1, 'Н': 2, 'Б': 0}   # opp_move -> optimal my_move idx

y_opp_all = df_raw['opp_move'].map(move_to_idx).values
groups = df_raw['match_id'].values

# ------------------------------------------------------------
# 1. Генерация признаков (единая)
# ------------------------------------------------------------
def create_features(df, global_opp_freq):
    df = df.copy()
    df['is_last_round'] = df['is_last_round'].astype(int)
    df['known_player'] = (df['player_name'] != 'Неизвестен').astype(int)

    win_cat_map = {'unknown': 0, 'до 5 побед': 1, '5-20': 2, '20-100': 3, '100+': 4}
    df['win_cat_code'] = df['win_category'].map(win_cat_map).fillna(0).astype(int)

    df['opp_wins_missing'] = (df['opp_match_wins'] == -1).astype(int)
    df['opp_winrate_missing'] = (df['opp_match_winrate'] == -0.01).astype(int)

    # Экспоненциальные частоты
    alpha = 0.7
    for n in [1, 2, 3, 4, 5, 6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in moves:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_opp_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'opp_exp_freq_{move}_last{n}'] = cnt

    # Биграмма
    df['opp_bigram'] = 'none'
    mask = df['prev2_opp_move'].notna() & df['prev1_opp_move'].notna()
    df.loc[mask, 'opp_bigram'] = (
        df.loc[mask, 'prev2_opp_move'].astype(str) + '_' +
        df.loc[mask, 'prev1_opp_move'].astype(str)
    )

    # Серийные признаки
    df['same_opp_as_prev1'] = (df['prev1_opp_move'] == df['prev2_opp_move']).astype(int)
    df['same_opp_as_prev2'] = (df['prev2_opp_move'] == df['prev3_opp_move']).astype(int)

    # Глобальные частоты
    for col in ['global_opp_К', 'global_opp_Н', 'global_opp_Б']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)
    df = df.join(global_opp_freq, on='player_name')
    df[['global_opp_К', 'global_opp_Н', 'global_opp_Б']] = \
        df[['global_opp_К', 'global_opp_Н', 'global_opp_Б']].fillna(0.0)

    return df

# ------------------------------------------------------------
# 2. Конфигурация
# ------------------------------------------------------------
exclude_cols = [
    'match_id', 'opp_move', 'my_move', 'outcome', 'player_name', 'win_category',
    'prev1_my_move', 'prev2_my_move', 'prev3_my_move',
    'prev4_my_move', 'prev5_my_move', 'prev6_my_move'
]

cat_features_base = [
    'stake', 'prev1_opp_move', 'prev2_opp_move', 'prev3_opp_move',
    'prev4_opp_move', 'prev5_opp_move', 'prev6_opp_move',
    'opp_bigram',
    'prev1_outcome', 'prev2_outcome', 'prev3_outcome',
    'prev4_outcome', 'prev5_outcome', 'prev6_outcome'
]

# ------------------------------------------------------------
# 3. Общая функция оценки
# ------------------------------------------------------------
def evaluate_model(model_name, predict_fn, n_splits=5):
    kf = GroupKFold(n_splits=n_splits)
    win_rates = []
    for tr_idx, val_idx in kf.split(df_raw, y_opp_all, groups):
        df_tr = df_raw.iloc[tr_idx].copy().reset_index(drop=True)
        df_val = df_raw.iloc[val_idx].copy().reset_index(drop=True)

        # Глобальные частоты только по train
        opp_counts = (df_tr.groupby('player_name')['opp_move']
                      .value_counts(normalize=True).unstack(fill_value=0))
        for m in moves:
            if m not in opp_counts.columns:
                opp_counts[m] = 0.0
        opp_counts = opp_counts[moves]
        opp_counts.columns = ['global_opp_К', 'global_opp_Н', 'global_opp_Б']

        X_tr_full = create_features(df_tr, opp_counts).reset_index(drop=True)
        X_val_full = create_features(df_val, opp_counts).reset_index(drop=True)

        feature_cols = [c for c in X_tr_full.columns if c not in exclude_cols]
        cat_features = [c for c in cat_features_base if c in feature_cols]

        X_tr = X_tr_full[feature_cols].copy()
        X_val = X_val_full[feature_cols].copy()

        # Преобразуем категориальные колонки в category для CatBoost / LightGBM
        for col in cat_features:
            X_tr[col] = X_tr[col].astype('category')
            X_val[col] = X_val[col].astype('category')

        y_tr_opp = y_opp_all[tr_idx]

        pred_optimal_idx = predict_fn(
            X_tr, y_tr_opp, X_val, cat_features,
            df_tr_feat=X_tr_full, df_val_feat=X_val_full,
            idx_tr=tr_idx, idx_val=val_idx
        )

        true_opt_idx = df_raw.iloc[val_idx]['opp_move'].map(optimal_map).values
        win = (pred_optimal_idx == true_opt_idx).mean()
        win_rates.append(win)

    return np.mean(win_rates), np.std(win_rates)

# ------------------------------------------------------------
# 4. Вспомогательная функция
# ------------------------------------------------------------
def probs_to_optimal_gain(prob_opp):
    p_k, p_n, p_b = prob_opp[:, 0], prob_opp[:, 1], prob_opp[:, 2]
    exp_k = p_n - p_b
    exp_n = p_b - p_k
    exp_b = p_k - p_n
    return np.argmax(np.column_stack([exp_k, exp_n, exp_b]), axis=1)

# ------------------------------------------------------------
# 5. Реализации моделей
# ------------------------------------------------------------

# 5.1 LightGBM базовые
def model_lgb_base(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    lgb = LGBMClassifier(objective='multiclass', n_estimators=200,
                         random_state=42, verbose=-1)
    lgb.fit(X_tr, y_tr_opp, categorical_feature=cat_feats)
    return probs_to_optimal_gain(lgb.predict_proba(X_val))

# 5.2 LightGBM + Optuna
def model_lgb_ext_optuna(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    lgb = LGBMClassifier(objective='multiclass', n_estimators=500,
                         learning_rate=0.02, num_leaves=31,
                         random_state=42, verbose=-1)
    lgb.fit(X_tr, y_tr_opp, categorical_feature=cat_feats)
    return probs_to_optimal_gain(lgb.predict_proba(X_val))

# 5.3 CatBoost расширенный (Trial 8)
def model_cb_ext_best(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    cb = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                            l2_leaf_reg=8.69, loss_function='MultiClass',
                            random_seed=42, verbose=False)
    cb.fit(X_tr, y_tr_opp, cat_features=cat_feats)
    return probs_to_optimal_gain(cb.predict_proba(X_val))

# 5.4 CatBoost + жёсткие правила
def model_cb_hard_rules(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    cb = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                            l2_leaf_reg=8.69, loss_function='MultiClass',
                            random_seed=42, verbose=False)
    cb.fit(X_tr, y_tr_opp, cat_features=cat_feats)
    proba = cb.predict_proba(X_val)

    df_val = kwargs['df_val_feat']
    delta = 0.15
    p = proba.copy()
    mask_draw = df_val['prev1_outcome'] == 'draw'
    for m, idx in move_to_idx.items():
        p[mask_draw & (df_val['prev1_opp_move'] == m), idx] += delta

    mask_twice = (df_val['prev1_opp_move'] == df_val['prev2_opp_move']) & df_val['prev1_opp_move'].notna()
    for m, idx in move_to_idx.items():
        p[mask_twice & (df_val['prev1_opp_move'] == m), idx] -= delta

    p = np.maximum(p, 0)
    p = p / p.sum(axis=1, keepdims=True)
    return probs_to_optimal_gain(p)

# 5.5 LinUCB (линейный бандит)
def model_linucb(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    idx_tr = kwargs['idx_tr']
    y_tr_opt = df_raw.iloc[idx_tr]['opp_move'].map(optimal_map).values

    num_cols = [c for c in X_tr.columns if c not in cat_feats]
    X_tr_cat = X_tr[cat_feats].astype(str)   # OneHotEncoder предпочитает строки
    X_val_cat = X_val[cat_feats].astype(str)

    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_enc = enc.fit_transform(X_tr_cat)
    X_val_enc = enc.transform(X_val_cat)

    X_tr_full = np.hstack([X_tr[num_cols].values, X_tr_enc])
    X_val_full = np.hstack([X_val[num_cols].values, X_val_enc])

    lr = LogisticRegression(solver='lbfgs', max_iter=1000)
    lr.fit(X_tr_full, y_tr_opt)
    return lr.predict(X_val_full).astype(int)

# 5.6 CatBoost + мета LR
def model_cb_meta_lr(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    df_tr_feat = kwargs['df_tr_feat']
    df_val_feat = kwargs['df_val_feat']

    inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    oof_proba = np.zeros((X_tr.shape[0], 3))
    for itr, iva in inner_cv.split(X_tr, y_tr_opp):
        X_itr, y_itr = X_tr.iloc[itr], y_tr_opp[itr]
        X_iva = X_tr.iloc[iva]
        cb = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                                l2_leaf_reg=8.69, loss_function='MultiClass',
                                random_seed=42, verbose=False)
        cb.fit(X_itr, y_itr, cat_features=cat_feats)
        oof_proba[iva] = cb.predict_proba(X_iva)

    cb_full = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                                 l2_leaf_reg=8.69, loss_function='MultiClass',
                                 random_seed=42, verbose=False)
    cb_full.fit(X_tr, y_tr_opp, cat_features=cat_feats)
    val_proba = cb_full.predict_proba(X_val)

    draw_rule = (df_tr_feat['prev1_outcome'] == 'draw').astype(int)
    avoid_rule = ((df_tr_feat['prev1_opp_move'] == df_tr_feat['prev2_opp_move']) &
                  df_tr_feat['prev1_opp_move'].notna()).astype(int)
    draw_rule_v = (df_val_feat['prev1_outcome'] == 'draw').astype(int)
    avoid_rule_v = ((df_val_feat['prev1_opp_move'] == df_val_feat['prev2_opp_move']) &
                    df_val_feat['prev1_opp_move'].notna()).astype(int)

    global_tr = df_tr_feat[['global_opp_К', 'global_opp_Н', 'global_opp_Б']].values
    global_val = df_val_feat[['global_opp_К', 'global_opp_Н', 'global_opp_Б']].values

    meta_tr = np.hstack([oof_proba,
                         draw_rule.values.reshape(-1, 1),
                         avoid_rule.values.reshape(-1, 1),
                         global_tr])
    meta_val = np.hstack([val_proba,
                          draw_rule_v.values.reshape(-1, 1),
                          avoid_rule_v.values.reshape(-1, 1),
                          global_val])

    lr_meta = LogisticRegression(solver='lbfgs', max_iter=1000, C=0.01)
    lr_meta.fit(meta_tr, y_tr_opp)
    return probs_to_optimal_gain(lr_meta.predict_proba(meta_val))

# 5.7 CatBoost + мета LGBM
def model_cb_meta_lgb(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    df_tr_feat = kwargs['df_tr_feat']
    df_val_feat = kwargs['df_val_feat']

    inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    oof_proba = np.zeros((X_tr.shape[0], 3))
    for itr, iva in inner_cv.split(X_tr, y_tr_opp):
        X_itr, y_itr = X_tr.iloc[itr], y_tr_opp[itr]
        X_iva = X_tr.iloc[iva]
        cb = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                                l2_leaf_reg=8.69, loss_function='MultiClass',
                                random_seed=42, verbose=False)
        cb.fit(X_itr, y_itr, cat_features=cat_feats)
        oof_proba[iva] = cb.predict_proba(X_iva)

    cb_full = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                                 l2_leaf_reg=8.69, loss_function='MultiClass',
                                 random_seed=42, verbose=False)
    cb_full.fit(X_tr, y_tr_opp, cat_features=cat_feats)
    val_proba = cb_full.predict_proba(X_val)

    draw_rule = (df_tr_feat['prev1_outcome'] == 'draw').astype(int)
    avoid_rule = ((df_tr_feat['prev1_opp_move'] == df_tr_feat['prev2_opp_move']) &
                  df_tr_feat['prev1_opp_move'].notna()).astype(int)
    draw_rule_v = (df_val_feat['prev1_outcome'] == 'draw').astype(int)
    avoid_rule_v = ((df_val_feat['prev1_opp_move'] == df_val_feat['prev2_opp_move']) &
                    df_val_feat['prev1_opp_move'].notna()).astype(int)

    global_tr = df_tr_feat[['global_opp_К', 'global_opp_Н', 'global_opp_Б']].values
    global_val = df_val_feat[['global_opp_К', 'global_opp_Н', 'global_opp_Б']].values

    meta_tr = np.hstack([oof_proba,
                         draw_rule.values.reshape(-1, 1),
                         avoid_rule.values.reshape(-1, 1),
                         global_tr])
    meta_val = np.hstack([val_proba,
                          draw_rule_v.values.reshape(-1, 1),
                          avoid_rule_v.values.reshape(-1, 1),
                          global_val])

    lgb_meta = LGBMClassifier(objective='multiclass', n_estimators=100,
                              num_leaves=5, learning_rate=0.02, verbose=-1)
    lgb_meta.fit(meta_tr, y_tr_opp)
    return probs_to_optimal_gain(lgb_meta.predict_proba(meta_val))

# 5.8 Блендинг LGB + LR + правило
def model_blend_lgb_lr_rule(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    df_val_feat = kwargs['df_val_feat']

    # LightGBM
    lgb = LGBMClassifier(objective='multiclass', n_estimators=300, learning_rate=0.03,
                         num_leaves=31, random_state=42, verbose=-1)
    lgb.fit(X_tr, y_tr_opp, categorical_feature=cat_feats)
    p_lgb = lgb.predict_proba(X_val)

    # LogisticRegression на one-hot
    num_cols = [c for c in X_tr.columns if c not in cat_feats]
    X_tr_cat = X_tr[cat_feats].astype(str)
    X_val_cat = X_val[cat_feats].astype(str)
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_enc = enc.fit_transform(X_tr_cat)
    X_val_enc = enc.transform(X_val_cat)
    X_tr_full = np.hstack([X_tr[num_cols].values, X_tr_enc])
    X_val_full = np.hstack([X_val[num_cols].values, X_val_enc])

    lr = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0)
    lr.fit(X_tr_full, y_tr_opp)
    p_lr = lr.predict_proba(X_val_full)

    # Правило «после ничьей бей ход соперника»
    rule_proba = np.ones_like(p_lgb) / 3.0
    mask_draw = df_val_feat['prev1_outcome'].values == 'draw'
    if mask_draw.any():
        # Индексы строк, где была ничья (позиционные, т.к. df_val_feat уже с reset_index)
        draw_indices = np.where(mask_draw)[0]
        opp_prev = df_val_feat.loc[mask_draw, 'prev1_opp_move'].map(move_to_idx).values
        # лучший ответ: 0->1, 1->2, 2->0
        best_idx = (opp_prev + 1) % 3
        for pos, b in zip(draw_indices, best_idx):
            rule_proba[pos, :] = 0.05
            rule_proba[pos, b] = 0.9

    mixed_proba = (p_lgb + p_lr + rule_proba) / 3.0
    return probs_to_optimal_gain(mixed_proba)

# 5.9 Аугментация (циклический сдвиг) CatBoost
def model_cb_augmented(X_tr, y_tr_opp, X_val, cat_feats, **kwargs):
    X_aug = X_tr.copy()
    y_aug = y_tr_opp.copy()

    move_cols = [c for c in X_tr.columns if 'move' in c or c == 'opp_bigram']

    rng = np.random.RandomState(42)
    mask_aug = rng.rand(len(X_aug)) < 0.5
    shift = {0: 1, 1: 2, 2: 0}

    for col in move_cols:
        if col in X_aug.columns:
            # Временно переводим в строку, затем в код, сдвигаем, обратно в строку
            codes = X_aug[col].astype(str).map(move_to_idx).fillna(-1).astype(int)
            codes[mask_aug] = codes[mask_aug].map(shift).fillna(-1).astype(int)
            # Обратно в категорию (как было), чтобы CatBoost воспринял
            X_aug[col] = codes.astype('category')

    # Для y_aug тоже сдвигаем маскированные
    y_aug_mask = y_aug.copy()
    y_aug_mask[mask_aug] = np.vectorize(shift.get)(y_aug_mask[mask_aug])

    X_tr_combined = pd.concat([X_tr, X_aug], ignore_index=True)
    y_tr_combined = np.concatenate([y_tr_opp, y_aug_mask])

    # Восстанавливаем категориальный тип для всех cat_features в объединённом трейне
    for col in cat_feats:
        if col in X_tr_combined.columns:
            X_tr_combined[col] = X_tr_combined[col].astype('category')
        if col in X_val.columns:
            X_val[col] = X_val[col].astype('category')

    cb = CatBoostClassifier(iterations=400, learning_rate=0.051, depth=4,
                            l2_leaf_reg=8.69, loss_function='MultiClass',
                            random_seed=42, verbose=False)
    cb.fit(X_tr_combined, y_tr_combined, cat_features=cat_feats)
    return probs_to_optimal_gain(cb.predict_proba(X_val))

# ------------------------------------------------------------
# 6. Сравнение всех моделей
# ------------------------------------------------------------
models = {
    "LightGBM (базовые)": model_lgb_base,
    "LightGBM + Optuna": model_lgb_ext_optuna,
    "CatBoost (Trial 8)": model_cb_ext_best,
    "CatBoost + жёсткие правила": model_cb_hard_rules,
    "LinUCB (линейный)": model_linucb,
    "CatBoost + мета LR": model_cb_meta_lr,
    "CatBoost + мета LGBM": model_cb_meta_lgb,
    "Блендинг LGB+LR+правило": model_blend_lgb_lr_rule,
    "Аугментация (CatBoost)": model_cb_augmented,
}

print("=" * 60)
print("Честное сравнение на 5-кратной кросс-валидации (GroupKFold)")
print("=" * 60)
results = {}
for name, fn in models.items():
    print(f"\n▶ Оценка модели: {name}")
    mean, std = evaluate_model(name, fn, n_splits=5)
    results[name] = (mean, std)
    print(f"   Win rate: {mean:.4f} ± {std:.4f}")

print("\n" + "=" * 60)
print("ИТОГИ:")
for name, (mean, std) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"{name:40s} | {mean:.4f} ± {std:.4f}")

Честное сравнение на 5-кратной кросс-валидации (GroupKFold)

▶ Оценка модели: LightGBM (базовые)
   Win rate: 0.3185 ± 0.0319

▶ Оценка модели: LightGBM + Optuna
   Win rate: 0.3245 ± 0.0280

▶ Оценка модели: CatBoost (Trial 8)
   Win rate: 0.2990 ± 0.0133

▶ Оценка модели: CatBoost + жёсткие правила
   Win rate: 0.3017 ± 0.0057

▶ Оценка модели: LinUCB (линейный)
   Win rate: 0.3581 ± 0.0184

▶ Оценка модели: CatBoost + мета LR
   Win rate: 0.2995 ± 0.0202

▶ Оценка модели: CatBoost + мета LGBM
   Win rate: 0.2919 ± 0.0150

▶ Оценка модели: Блендинг LGB+LR+правило
   Win rate: 0.3185 ± 0.0217

▶ Оценка модели: Аугментация (CatBoost)
   Win rate: 0.3076 ± 0.0189

ИТОГИ:
LinUCB (линейный)                        | 0.3581 ± 0.0184
LightGBM + Optuna                        | 0.3245 ± 0.0280
Блендинг LGB+LR+правило                  | 0.3185 ± 0.0217
LightGBM (базовые)                       | 0.3185 ± 0.0319
Аугментация (CatBoost)                   | 0.3076 ± 0.0189
CatBoost + жёсткие правила

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0. Данные и константы
# ------------------------------------------------------------
df_raw = pd.read_csv('rps_data.csv')
moves = ['К', 'Н', 'Б']
move_to_idx = {'К': 0, 'Н': 1, 'Б': 2}
optimal_map = {'К': 1, 'Н': 2, 'Б': 0}

y_opp_all = df_raw['opp_move'].map(move_to_idx).values
# Истинный оптимальный ход (какой нужно поставить, чтобы выиграть)
y_opt_all = df_raw['opp_move'].map(optimal_map).values
groups = df_raw['match_id'].values

# ------------------------------------------------------------
# 1. Генерация признаков (добавлен prev_my_move)
# ------------------------------------------------------------
def create_features(df, global_opp_freq):
    df = df.copy()
    df['is_last_round'] = df['is_last_round'].astype(int)
    df['known_player'] = (df['player_name'] != 'Неизвестен').astype(int)

    win_cat_map = {'unknown': 0, 'до 5 побед': 1, '5-20': 2, '20-100': 3, '100+': 4}
    df['win_cat_code'] = df['win_category'].map(win_cat_map).fillna(0).astype(int)
    df['opp_wins_missing'] = (df['opp_match_wins'] == -1).astype(int)
    df['opp_winrate_missing'] = (df['opp_match_winrate'] == -0.01).astype(int)

    # Экспоненциальные частоты ходов соперника (без изменений)
    alpha = 0.7
    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in moves:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_opp_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'opp_exp_freq_{move}_last{n}'] = cnt

    # Частоты своих ходов (новые!)
    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in moves:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_my_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'my_exp_freq_{move}_last{n}'] = cnt

    # Биграмма соперника (prev2_opp_move + prev1_opp_move)
    df['opp_bigram'] = 'none'
    mask = df['prev2_opp_move'].notna() & df['prev1_opp_move'].notna()
    df.loc[mask, 'opp_bigram'] = (
        df.loc[mask, 'prev2_opp_move'].astype(str) + '_' +
        df.loc[mask, 'prev1_opp_move'].astype(str)
    )

    # Биграмма своих ходов (новая!)
    df['my_bigram'] = 'none'
    mask_my = df['prev2_my_move'].notna() & df['prev1_my_move'].notna()
    df.loc[mask_my, 'my_bigram'] = (
        df.loc[mask_my, 'prev2_my_move'].astype(str) + '_' +
        df.loc[mask_my, 'prev1_my_move'].astype(str)
    )

    # Серийные признаки (повторения)
    df['same_opp_as_prev1'] = (df['prev1_opp_move'] == df['prev2_opp_move']).astype(int)
    df['same_my_as_prev1'] = (df['prev1_my_move'] == df['prev2_my_move']).astype(int)

    # Глобальные частоты соперника
    for col in ['global_opp_К','global_opp_Н','global_opp_Б']:
        if col in df.columns: df.drop(columns=col, inplace=True)
    df = df.join(global_opp_freq, on='player_name')
    df[['global_opp_К','global_opp_Н','global_opp_Б']] = \
        df[['global_opp_К','global_opp_Н','global_opp_Б']].fillna(0.0)

    return df

# ------------------------------------------------------------
# 2. Исключаемые колонки (теперь my_move тоже исключается, но prev_my_move используются в фичах)
# ------------------------------------------------------------
exclude_cols = [
    'match_id', 'opp_move', 'my_move', 'outcome', 'player_name', 'win_category'
    # prevN_my_move НЕ исключаем, т.к. они нужны для генерации фич, но сами столбцы уже превращены в фичи
]
# Сами столбцы prevN_my_move попадут в данные, но мы их потом дропнем из финальных признаков,
# т.к. мы уже извлекли из них признаки. Поэтому дополним exclude_cols ниже при отборе.

# ------------------------------------------------------------
# 3. Оценка одной модели с кастомной функцией predict
# ------------------------------------------------------------
def evaluate_model(name, predict_fn, n_splits=5):
    kf = GroupKFold(n_splits=n_splits)
    wins = []
    for tr_idx, val_idx in kf.split(df_raw, y_opt_all, groups):
        df_tr = df_raw.iloc[tr_idx].copy().reset_index(drop=True)
        df_val = df_raw.iloc[val_idx].copy().reset_index(drop=True)

        # Глобальные частоты соперника из трейна
        opp_cnt = (df_tr.groupby('player_name')['opp_move']
                   .value_counts(normalize=True).unstack(fill_value=0))
        for m in moves:
            if m not in opp_cnt.columns: opp_cnt[m] = 0.0
        opp_cnt = opp_cnt[moves]
        opp_cnt.columns = ['global_opp_К','global_opp_Н','global_opp_Б']

        X_tr_full = create_features(df_tr, opp_cnt).reset_index(drop=True)
        X_val_full = create_features(df_val, opp_cnt).reset_index(drop=True)

        # Дополнительно исключаем оригинальные prevN_my_move (они уже в фичах)
        extra_exclude = [c for c in X_tr_full.columns if 'prev' in c and 'my_move' in c and c in df_raw.columns]
        feature_cols = [c for c in X_tr_full.columns if c not in exclude_cols + extra_exclude]
        cat_feats = [c for c in ['stake','prev1_opp_move','prev2_opp_move','prev3_opp_move',
                                 'prev4_opp_move','prev5_opp_move','prev6_opp_move',
                                 'opp_bigram','my_bigram',
                                 'prev1_outcome','prev2_outcome','prev3_outcome',
                                 'prev4_outcome','prev5_outcome','prev6_outcome']
                     if c in feature_cols]

        X_tr = X_tr_full[feature_cols].copy()
        X_val = X_val_full[feature_cols].copy()
        for col in cat_feats:
            X_tr[col] = X_tr[col].astype('category')
            X_val[col] = X_val[col].astype('category')

        y_tr_opt = y_opt_all[tr_idx]
        pred = predict_fn(X_tr, y_tr_opt, X_val, cat_feats,
                          df_tr_feat=X_tr_full, df_val_feat=X_val_full,
                          idx_tr=tr_idx, idx_val=val_idx)
        true_opt = y_opt_all[val_idx]
        wins.append((pred == true_opt).mean())
    return np.mean(wins), np.std(wins)

# ------------------------------------------------------------
# 4. Новая модель: CatBoost на y_opt с подбором гиперпараметров под win-rate
# ------------------------------------------------------------
def objective(trial, X_train, y_train, groups_train, cat_features):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': 42,
        'loss_function': 'MultiClass',
        'verbose': False,
        'allow_writing_files': False
    }
    # Внутренняя 3-фолдовая групповая валидация
    inner_kf = GroupKFold(n_splits=3)
    win_rates = []
    for itr, iva in inner_kf.split(X_train, y_train, groups_train):
        X_itr, y_itr = X_train.iloc[itr], y_train[itr]
        X_iva, y_iva = X_train.iloc[iva], y_train[iva]
        cb = CatBoostClassifier(**params)
        cb.fit(X_itr, y_itr, cat_features=cat_features)
        pred = cb.predict(X_iva).flatten().astype(int)
        win_rates.append((pred == y_iva).mean())
    return np.mean(win_rates)

# Функция-обёртка для predict_fn
class DirectOptimalCB:
    def __init__(self, n_trials=25):
        self.n_trials = n_trials

    def __call__(self, X_tr, y_tr_opt, X_val, cat_feats, **kwargs):
        # Подбираем гиперпараметры на трейне с учётом групп
        idx_tr = kwargs['idx_tr']
        groups_tr = groups[idx_tr]
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda trial: objective(trial, X_tr, y_tr_opt, groups_tr, cat_feats),
                       n_trials=self.n_trials, show_progress_bar=False)
        best_params = study.best_params
        # Обучаем лучшую модель на всём трейне
        cb = CatBoostClassifier(
            iterations=best_params['iterations'],
            learning_rate=best_params['learning_rate'],
            depth=best_params['depth'],
            l2_leaf_reg=best_params['l2_leaf_reg'],
            loss_function='MultiClass',
            random_seed=42, verbose=False
        )
        cb.fit(X_tr, y_tr_opt, cat_features=cat_feats)
        return cb.predict(X_val).flatten().astype(int)

# 5. Ансамбль DirectCatBoost + LinUCB (усреднение решений)
def ensemble_direct_cb_linucb(X_tr, y_tr_opt, X_val, cat_feats, **kwargs):
    # Direct CatBoost (Optuna)
    cb_model = DirectOptimalCB(n_trials=20)
    pred_cb = cb_model(X_tr, y_tr_opt, X_val, cat_feats, **kwargs)

    # LinUCB (аналогично model_linucb, но на y_opt)
    idx_tr = kwargs['idx_tr']
    num_cols = [c for c in X_tr.columns if c not in cat_feats]
    X_tr_cat = X_tr[cat_feats].astype(str)
    X_val_cat = X_val[cat_feats].astype(str)
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_enc = enc.fit_transform(X_tr_cat)
    X_val_enc = enc.transform(X_val_cat)
    X_tr_full = np.hstack([X_tr[num_cols].values, X_tr_enc])
    X_val_full = np.hstack([X_val[num_cols].values, X_val_enc])
    lr = LogisticRegression(solver='lbfgs', max_iter=1000)
    lr.fit(X_tr_full, y_tr_opt)
    pred_lr = lr.predict(X_val_full).astype(int)

    # Простое голосование: если модели согласны — ок, иначе случайный выбор (или выбираем первую)
    # Лучше: смешиваем через вероятности? Но LinUCB вероятностей не даёт (если не predict_proba).
    # Используем predict_proba у LR (для трёх классов) и у CB, усредняем и берём argmax.
    proba_cb = CatBoostClassifier(random_seed=42, verbose=False)  # придётся обучить заново для вероятностей
    # Чтобы не переобучать, сделаем внутри: быстро обучим CB на X_tr, y_tr_opt с лучшими параметрами? 
    # Упростим: если предсказания совпали, берём его, иначе берём предсказание LR.
    pred_ensemble = np.where(pred_cb == pred_lr, pred_cb, pred_lr)
    return pred_ensemble

# ------------------------------------------------------------
# 6. Сравнение: добавим новую модель и ансамбль
# ------------------------------------------------------------
models = {
    "DirectCatBoost (Optuna)": DirectOptimalCB(n_trials=20),
    "Ensemble DirectCB+LinUCB": ensemble_direct_cb_linucb,
    # Оставляем лучшего из предыдущих для сравнения
    "LinUCB (линейный)": model_linucb,   # должна быть определена, но для примера оставим
}

# ВАЖНО: model_linucb из предыдущего скрипта нужно добавить в код.
# Я приведу её кратко:

def model_linucb(X_tr, y_tr_opp_unused, X_val, cat_feats, **kwargs):
    idx_tr = kwargs['idx_tr']
    y_tr_opt = df_raw.iloc[idx_tr]['opp_move'].map(optimal_map).values
    num_cols = [c for c in X_tr.columns if c not in cat_feats]
    X_tr_cat = X_tr[cat_feats].astype(str)
    X_val_cat = X_val[cat_feats].astype(str)
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_tr_enc = enc.fit_transform(X_tr_cat)
    X_val_enc = enc.transform(X_val_cat)
    X_tr_full = np.hstack([X_tr[num_cols].values, X_tr_enc])
    X_val_full = np.hstack([X_val[num_cols].values, X_val_enc])
    lr = LogisticRegression(solver='lbfgs', max_iter=1000)
    lr.fit(X_tr_full, y_tr_opt)
    return lr.predict(X_val_full).astype(int)

# ------------------------------------------------------------
# 7. Запуск сравнения
# ------------------------------------------------------------
if __name__ == "__main__":
    print("Оценка улучшенных моделей (GroupKFold, 5 фолдов)...")
    for name, fn in models.items():
        mean, std = evaluate_model(name, fn, n_splits=5)
        print(f"{name}: {mean:.4f} ± {std:.4f}")

Оценка улучшенных моделей (GroupKFold, 5 фолдов)...


[I 2026-04-26 12:06:44,334] A new study created in memory with name: no-name-69b15610-49b1-4a6f-9095-6e9aa53ba2d1
[I 2026-04-26 12:07:27,104] Trial 0 finished with value: 0.4267175003725597 and parameters: {'iterations': 350, 'learning_rate': 0.08927180304353628, 'depth': 5, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.4267175003725597.
[I 2026-04-26 12:07:41,536] Trial 1 finished with value: 0.44978860684737193 and parameters: {'iterations': 262, 'learning_rate': 0.01432169828911152, 'depth': 3, 'l2_leaf_reg': 8.795585311974417}. Best is trial 1 with value: 0.44978860684737193.
[I 2026-04-26 12:08:08,607] Trial 2 finished with value: 0.4375838259400924 and parameters: {'iterations': 441, 'learning_rate': 0.051059032093947576, 'depth': 3, 'l2_leaf_reg': 9.72918866945795}. Best is trial 1 with value: 0.44978860684737193.
[I 2026-04-26 12:08:36,324] Trial 3 finished with value: 0.4518225070234409 and parameters: {'iterations': 533, 'learning_rate': 0.01630568734622147

DirectCatBoost (Optuna): 0.3706 ± 0.0119


[I 2026-04-26 13:28:57,743] A new study created in memory with name: no-name-4509fa43-3a82-47e5-9af5-5c7818365290
[I 2026-04-26 13:30:07,460] Trial 0 finished with value: 0.4267175003725597 and parameters: {'iterations': 350, 'learning_rate': 0.08927180304353628, 'depth': 5, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.4267175003725597.
[I 2026-04-26 13:30:31,108] Trial 1 finished with value: 0.44978860684737193 and parameters: {'iterations': 262, 'learning_rate': 0.01432169828911152, 'depth': 3, 'l2_leaf_reg': 8.795585311974417}. Best is trial 1 with value: 0.44978860684737193.
[I 2026-04-26 13:31:10,763] Trial 2 finished with value: 0.4375838259400924 and parameters: {'iterations': 441, 'learning_rate': 0.051059032093947576, 'depth': 3, 'l2_leaf_reg': 9.72918866945795}. Best is trial 1 with value: 0.44978860684737193.
[I 2026-04-26 13:32:00,205] Trial 3 finished with value: 0.4518225070234409 and parameters: {'iterations': 533, 'learning_rate': 0.01630568734622147

Ensemble DirectCB+LinUCB: 0.3646 ± 0.0287
LinUCB (линейный): 0.3646 ± 0.0287


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0. Данные и константы
# ------------------------------------------------------------
df_raw = pd.read_csv('rps_data.csv')
moves = ['К', 'Н', 'Б']
move_to_idx = {'К': 0, 'Н': 1, 'Б': 2}
optimal_map = {'К': 1, 'Н': 2, 'Б': 0}

y_opt_all = df_raw['opp_move'].map(optimal_map).values
groups = df_raw['match_id'].values

# ------------------------------------------------------------
# 1. Генератор признаков (тот же расширенный)
# ------------------------------------------------------------
def create_features(df, global_opp_freq):
    df = df.copy()
    df['is_last_round'] = df['is_last_round'].astype(int)
    df['known_player'] = (df['player_name'] != 'Неизвестен').astype(int)
    win_cat_map = {'unknown': 0, 'до 5 побед': 1, '5-20': 2, '20-100': 3, '100+': 4}
    df['win_cat_code'] = df['win_category'].map(win_cat_map).fillna(0).astype(int)
    df['opp_wins_missing'] = (df['opp_match_wins'] == -1).astype(int)
    df['opp_winrate_missing'] = (df['opp_match_winrate'] == -0.01).astype(int)

    alpha = 0.7
    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in moves:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_opp_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'opp_exp_freq_{move}_last{n}'] = cnt

    for outcome in ['win', 'lose', 'draw']:
        mask_outcome = df['prev1_outcome'] == outcome
        for move in moves:
            col = f'opp_freq_{move}_after_{outcome}'
            df[col] = 0.0
            if mask_outcome.any():
                df.loc[mask_outcome, col] = (df.loc[mask_outcome, 'prev1_opp_move'] == move).astype(int)

    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in moves:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_my_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'my_exp_freq_{move}_last{n}'] = cnt

    df['opp_bigram'] = 'none'
    mask = df['prev2_opp_move'].notna() & df['prev1_opp_move'].notna()
    df.loc[mask, 'opp_bigram'] = df.loc[mask, 'prev2_opp_move'].astype(str)+'_'+df.loc[mask, 'prev1_opp_move'].astype(str)

    df['my_bigram'] = 'none'
    mask_my = df['prev2_my_move'].notna() & df['prev1_my_move'].notna()
    df.loc[mask_my, 'my_bigram'] = df.loc[mask_my, 'prev2_my_move'].astype(str)+'_'+df.loc[mask_my, 'prev1_my_move'].astype(str)

    df['triple'] = 'none'
    mask_tr = df['prev1_opp_move'].notna() & df['prev1_my_move'].notna() & df['prev1_outcome'].notna()
    df.loc[mask_tr, 'triple'] = (df.loc[mask_tr, 'prev1_opp_move'].astype(str) + '_' +
                                 df.loc[mask_tr, 'prev1_my_move'].astype(str) + '_' +
                                 df.loc[mask_tr, 'prev1_outcome'].astype(str))

    df['same_opp_as_prev1'] = (df['prev1_opp_move'] == df['prev2_opp_move']).astype(int)
    df['same_my_as_prev1'] = (df['prev1_my_move'] == df['prev2_my_move']).astype(int)

    df['score_diff'] = df['score_me_before'] - df['score_opp_before']
    df['stake_num'] = df['stake'].map({'25': 25, '50': 50, '100': 100}).fillna(50)

    for col in ['global_opp_К','global_opp_Н','global_opp_Б']:
        if col in df.columns: df.drop(columns=col, inplace=True)
    df = df.join(global_opp_freq, on='player_name')
    df[['global_opp_К','global_opp_Н','global_opp_Б']] = df[['global_opp_К','global_opp_Н','global_opp_Б']].fillna(0.0)

    return df

exclude_cols = [
    'match_id', 'opp_move', 'my_move', 'outcome', 'player_name', 'win_category',
    'prev1_my_move', 'prev2_my_move', 'prev3_my_move',
    'prev4_my_move', 'prev5_my_move', 'prev6_my_move',
    'score_me_before', 'score_opp_before'
]
cat_features_base = [
    'stake',
    'prev1_opp_move', 'prev2_opp_move', 'prev3_opp_move',
    'prev4_opp_move', 'prev5_opp_move', 'prev6_opp_move',
    'opp_bigram', 'my_bigram', 'triple',
    'prev1_outcome', 'prev2_outcome', 'prev3_outcome',
    'prev4_outcome', 'prev5_outcome', 'prev6_outcome'
]

# ------------------------------------------------------------
# 2. Функция оценки
# ------------------------------------------------------------
def evaluate_model(name, predict_fn, n_splits=5):
    kf = GroupKFold(n_splits=n_splits)
    wins = []
    for tr_idx, val_idx in kf.split(df_raw, y_opt_all, groups):
        df_tr = df_raw.iloc[tr_idx].copy().reset_index(drop=True)
        df_val = df_raw.iloc[val_idx].copy().reset_index(drop=True)

        opp_cnt = (df_tr.groupby('player_name')['opp_move']
                   .value_counts(normalize=True).unstack(fill_value=0))
        for m in moves:
            if m not in opp_cnt.columns: opp_cnt[m] = 0.0
        opp_cnt = opp_cnt[moves]
        opp_cnt.columns = ['global_opp_К','global_opp_Н','global_opp_Б']

        X_tr_full = create_features(df_tr, opp_cnt).reset_index(drop=True)
        X_val_full = create_features(df_val, opp_cnt).reset_index(drop=True)

        feature_cols = [c for c in X_tr_full.columns if c not in exclude_cols]
        cat_feats = [c for c in cat_features_base if c in feature_cols]

        X_tr = X_tr_full[feature_cols]
        X_val = X_val_full[feature_cols]
        for col in cat_feats:
            X_tr[col] = X_tr[col].astype('category')
            X_val[col] = X_val[col].astype('category')

        y_tr_opt = y_opt_all[tr_idx]
        pred = predict_fn(X_tr, y_tr_opt, X_val, cat_feats,
                          df_tr_feat=X_tr_full, df_val_feat=X_val_full)
        true_opt = y_opt_all[val_idx]
        wins.append((pred == true_opt).mean())
    return np.mean(wins), np.std(wins)

# ------------------------------------------------------------
# 3. Модель: ансамбль CatBoost + калибровка + правила
# ------------------------------------------------------------
CB_PARAMS = {
    'iterations': 500, 'learning_rate': 0.03, 'depth': 4,
    'loss_function': 'MultiClass', 'verbose': False
}
N_ENSEMBLE = 3

def model_ensemble_cb_calibrated_rules(X_tr, y_tr_opt, X_val, cat_feats, **kwargs):
    df_tr_feat = kwargs['df_tr_feat']
    df_val_feat = kwargs['df_val_feat']

    # Обучаем N моделей с разными random_seed
    models = []
    probs_tr = np.zeros((X_tr.shape[0], 3))
    probs_val = np.zeros((X_val.shape[0], 3))
    for i in range(N_ENSEMBLE):
        cb = CatBoostClassifier(random_seed=42 + i*10, **CB_PARAMS)
        cb.fit(X_tr, y_tr_opt, cat_features=cat_feats)
        models.append(cb)
        probs_tr += cb.predict_proba(X_tr)
        probs_val += cb.predict_proba(X_val)
    probs_tr /= N_ENSEMBLE
    probs_val /= N_ENSEMBLE

    # Добавляем глобальные частоты как дополнительные фичи для калибратора
    global_tr = df_tr_feat[['global_opp_К','global_opp_Н','global_opp_Б']].values
    global_val = df_val_feat[['global_opp_К','global_opp_Н','global_opp_Б']].values

    cal_tr = np.hstack([probs_tr, global_tr])
    cal_val = np.hstack([probs_val, global_val])

    cal = LogisticRegression(solver='lbfgs', max_iter=1000)
    cal.fit(cal_tr, y_tr_opt)
    cal_probs = cal.predict_proba(cal_val)

    # Получаем предварительные предсказания (индексы)
    raw_preds = np.argmax(cal_probs, axis=1)   # 0,1,2

    # --- Правила, переопределяющие решение в особых случаях ---
    df_val = df_val_feat
    preds = raw_preds.copy()
    valid_opp = df_val['prev1_opp_move'].isin(moves).values

    # Правило 1: после ничьей — бей ход соперника (сильный сигнал)
    mask_draw = (df_val['prev1_outcome'] == 'draw').values & valid_opp
    if mask_draw.any():
        opp_idx = df_val.loc[mask_draw, 'prev1_opp_move'].map(move_to_idx).values
        best_resp = (opp_idx + 1) % 3
        preds[mask_draw] = best_resp

    # Правило 2: соперник повторил ход дважды — избегаем бить этот ход (ставим любой другой)
    mask_twice = ((df_val['prev1_opp_move'] == df_val['prev2_opp_move']).values &
                  valid_opp & df_val['prev2_opp_move'].isin(moves).values)
    if mask_twice.any():
        opp_idx = df_val.loc[mask_twice, 'prev1_opp_move'].map(move_to_idx).values
        # Ход, бьющий opp_idx, НЕ ставим
        # Определяем новый ход: (opp_idx + 2) % 3  (это будет третий вариант, не opp_idx и не бьющий)
        alternative = (opp_idx + 2) % 3
        preds[mask_twice] = alternative

    return preds

# ------------------------------------------------------------
# 4. Запуск
# ------------------------------------------------------------
models = {
    "Ensemble CB + Calibration + Rules": model_ensemble_cb_calibrated_rules,
}

print("Честная групповая кросс-валидация (5 фолдов)...\n")
results = {}
for name, fn in models.items():
    mean, std = evaluate_model(name, fn)
    results[name] = (mean, std)
    print(f"{name:40s} | Win rate: {mean:.4f} ± {std:.4f}")

# Для сравнения добавим одиночный CatBoost
print("\nДобавляем одиночный CatBoost (Direct)...")
def model_single_cb(X_tr, y_tr_opt, X_val, cat_feats, **kwargs):
    cb = CatBoostClassifier(random_seed=42, **CB_PARAMS)
    cb.fit(X_tr, y_tr_opt, cat_features=cat_feats)
    return cb.predict(X_val).flatten().astype(int)

mean_single, std_single = evaluate_model("Single CatBoost (direct)", model_single_cb)
print(f"{'Single CatBoost (direct)':40s} | Win rate: {mean_single:.4f} ± {std_single:.4f}")

Честная групповая кросс-валидация (5 фолдов)...

Ensemble CB + Calibration + Rules        | Win rate: 0.3652 ± 0.0215

Добавляем одиночный CatBoost (Direct)...
Single CatBoost (direct)                 | Win rate: 0.3668 ± 0.0111


In [37]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

# ------------------------------------------------------------
# Параметры CatBoost
# ------------------------------------------------------------
CB_PARAMS = {
    'iterations': 500,
    'learning_rate': 0.03,
    'depth': 4,
    'loss_function': 'MultiClass',
    'random_seed': 42,
    'verbose': False
}

# ------------------------------------------------------------
# Генератор признаков
# ------------------------------------------------------------
def create_features(df, global_opp_freq=None):
    df = df.copy()
    df['is_last_round'] = df['is_last_round'].astype(int)
    df['known_player'] = (df['player_name'] != 'Неизвестен').astype(int)
    win_cat_map = {'unknown': 0, 'до 5 побед': 1, '5-20': 2, '20-100': 3, '100+': 4}
    df['win_cat_code'] = df['win_category'].map(win_cat_map).fillna(0).astype(int)
    df['opp_wins_missing'] = (df['opp_match_wins'] == -1).astype(int)
    df['opp_winrate_missing'] = (df['opp_match_winrate'] == -0.01).astype(int)

    alpha = 0.7
    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in ['К','Н','Б']:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_opp_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'opp_exp_freq_{move}_last{n}'] = cnt

    for n in [1,2,3,4,5,6]:
        weights = np.array([alpha**i for i in range(n)])
        weights = weights / weights.sum()
        for move in ['К','Н','Б']:
            cnt = np.zeros(len(df))
            for i in range(n):
                prev_col = f'prev{i+1}_my_move'
                if prev_col in df.columns:
                    cnt += (df[prev_col] == move).astype(int) * weights[i]
            df[f'my_exp_freq_{move}_last{n}'] = cnt

    df['opp_bigram'] = 'none'
    mask = df['prev2_opp_move'].notna() & df['prev1_opp_move'].notna()
    df.loc[mask, 'opp_bigram'] = df.loc[mask, 'prev2_opp_move'].astype(str)+'_'+df.loc[mask, 'prev1_opp_move'].astype(str)
    df['my_bigram'] = 'none'
    mask_my = df['prev2_my_move'].notna() & df['prev1_my_move'].notna()
    df.loc[mask_my, 'my_bigram'] = df.loc[mask_my, 'prev2_my_move'].astype(str)+'_'+df.loc[mask_my, 'prev1_my_move'].astype(str)
    df['triple'] = 'none'
    mask_tr = df['prev1_opp_move'].notna() & df['prev1_my_move'].notna() & df['prev1_outcome'].notna()
    df.loc[mask_tr, 'triple'] = (df.loc[mask_tr, 'prev1_opp_move'].astype(str) + '_' +
                                 df.loc[mask_tr, 'prev1_my_move'].astype(str) + '_' +
                                 df.loc[mask_tr, 'prev1_outcome'].astype(str))
    df['same_opp_as_prev1'] = (df['prev1_opp_move'] == df['prev2_opp_move']).astype(int)
    df['same_my_as_prev1'] = (df['prev1_my_move'] == df['prev2_my_move']).astype(int)
    df['score_diff'] = df['score_me_before'] - df['score_opp_before']

    if global_opp_freq is not None:
        for col in ['global_opp_К','global_opp_Н','global_opp_Б']:
            if col in df.columns: df.drop(columns=col, inplace=True)
        df = df.join(global_opp_freq, on='player_name')
        df[['global_opp_К','global_opp_Н','global_opp_Б']] = \
            df[['global_opp_К','global_opp_Н','global_opp_Б']].fillna(0.0)
    return df

# ------------------------------------------------------------
# Категориальные признаки
# ------------------------------------------------------------
CAT_FEATURES = [
    'stake',
    'prev1_opp_move', 'prev2_opp_move', 'prev3_opp_move',
    'prev4_opp_move', 'prev5_opp_move', 'prev6_opp_move',
    'opp_bigram', 'my_bigram', 'triple',
    'prev1_outcome', 'prev2_outcome', 'prev3_outcome',
    'prev4_outcome', 'prev5_outcome', 'prev6_outcome'
]

# ------------------------------------------------------------
# Walk-forward backtest (исправленный)
# ------------------------------------------------------------
def backtest_catboost(data_path='rps_data.csv'):
    df = pd.read_csv(data_path)
    numeric_cols = ['match_id', 'round', 'opp_match_wins', 'opp_match_winrate', 'stake',
                    'score_me_before', 'score_opp_before', 'score_diff', 'streak_draws', 'is_last_round']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    def compute_win_category(wins):
        if wins == -1: return 'unknown'
        elif wins <= 5: return '<=5'
        elif wins <= 20: return '6-20'
        elif wins <= 100: return '21-100'
        else: return '>100'
    if 'win_category' not in df.columns:
        df['win_category'] = df['opp_match_wins'].apply(compute_win_category)

    for col in df.columns:
        if 'prev' in col and 'outcome' in col:
            df[col] = df[col].astype(str)

    df = df.sort_values(['match_id', 'round']).reset_index(drop=True)

    optimal_map = {'К': 'Н', 'Н': 'Б', 'Б': 'К'}
    df['optimal_move'] = df['opp_move'].map(optimal_map)

    exclude = ['match_id', 'opp_move', 'my_move', 'outcome', 'player_name', 'win_category',
               'prev1_my_move', 'prev2_my_move', 'prev3_my_move',
               'prev4_my_move', 'prev5_my_move', 'prev6_my_move',
               'score_me_before', 'score_opp_before', 'optimal_move']
    feature_cols = [c for c in df.columns if c not in exclude]

    print("Walk-forward backtest с CatBoost (исправленный)...")
    history_db = pd.DataFrame(columns=df.columns)
    buffer = []
    current_mid = None
    model = None
    preds = []
    trues = []

    for idx, row in df.iterrows():
        mid = row['match_id']
        if mid != current_mid:
            if buffer:
                df_match = pd.DataFrame(buffer)
                df_match = df_match[history_db.columns]
                history_db = pd.concat([history_db, df_match], ignore_index=True)
                buffer = []
            current_mid = mid
            model = None
            if not history_db.empty:
                train_df = create_features(history_db.copy())
                X_train = train_df[feature_cols].copy()
                for c in CAT_FEATURES:
                    if c in X_train.columns:
                        X_train[c] = X_train[c].astype(str)
                cat_feats_train = [c for c in CAT_FEATURES if c in X_train.columns]
                y_train = history_db['optimal_move'].map({'К':0, 'Н':1, 'Б':2}).values
                model = CatBoostClassifier(**CB_PARAMS)
                model.fit(X_train, y_train, cat_features=cat_feats_train)

        if model is not None:
            single_row_df = pd.DataFrame([row.to_dict()])
            single_feat = create_features(single_row_df)
            X_pred = single_feat[feature_cols].copy()
            for c in CAT_FEATURES:
                if c in X_pred.columns:
                    X_pred[c] = X_pred[c].astype(str)
            X_pred = X_pred[list(X_train.columns)]
            pred_idx = model.predict(X_pred)[0].item()   # исправлено: извлекаем скаляр
            idx_to_move = {0: 'К', 1: 'Н', 2: 'Б'}
            pred_move = idx_to_move[pred_idx]

            if buffer:
                last = buffer[-1]
                last_outcome = last['outcome']
                last_opp = last['opp_move']
                if last_outcome == 'draw' and last_opp in ['К','Н','Б']:
                    beat = {'К': 'Н', 'Н': 'Б', 'Б': 'К'}
                    pred_move = beat[last_opp]
                elif len(buffer) >= 2:
                    prev2_opp = buffer[-2]['opp_move']
                    if prev2_opp == last_opp:
                        beat = {'К': 'Н', 'Н': 'Б', 'Б': 'К'}
                        avoid = beat[last_opp]
                        alternatives = [m for m in ['К','Н','Б'] if m != avoid]
                        if pred_move == avoid:
                            pred_move = alternatives[0]
        else:
            pred_move = 'К'

        preds.append(pred_move)
        trues.append(row['optimal_move'])
        buffer.append(row.to_dict())

    if buffer:
        df_match = pd.DataFrame(buffer)
        df_match = df_match[history_db.columns]
        history_db = pd.concat([history_db, df_match], ignore_index=True)

    win_rate = np.mean(np.array(preds) == np.array(trues))
    print(f"Честный win rate CatBoost на исторических данных: {win_rate:.4f} ({len(preds)} раундов)")
    return win_rate

if __name__ == "__main__":
    backtest_catboost('rps_data.csv')

Walk-forward backtest с CatBoost (исправленный)...
Честный win rate CatBoost на исторических данных: 0.3592 (1843 раундов)


In [29]:
df_ml = df[df['round'] > 2].copy()

In [30]:
df_ml

,match_id,round,player_name,win_category,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,outcome,...,prev4_opp_move,prev4_my_move,prev4_outcome,prev5_opp_move,prev5_my_move,prev5_outcome,prev6_opp_move,prev6_my_move,prev6_outcome,next_opp_move
2,1,3,Неизвестен,unknown,-1,-0.01,25,Н,Н,draw,...,-1,-1,none,-1,-1,none,-1,-1,none,Н
3,1,4,Неизвестен,unknown,-1,-0.01,25,Н,Н,draw,...,-1,-1,none,-1,-1,none,-1,-1,none,Н
4,1,5,Неизвестен,unknown,-1,-0.01,25,Н,К,win,...,Н,Б,lose,-1,-1,none,-1,-1,none,Н
8,2,3,Неизвестен,unknown,-1,-0.01,25,Б,Н,win,...,-1,-1,none,-1,-1,none,-1,-1,none,Б
9,2,4,Неизвестен,unknown,-1,-0.01,25,Б,К,lose,...,-1,-1,none,-1,-1,none,-1,-1,none,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1834,289,4,Громовой Дубль,<=5,1,1.00,50,К,К,draw,...,-1,-1,none,-1,-1,none,-1,-1,none,К
1835,289,5,Громовой Дубль,<=5,1,1.00,50,К,Н,lose,...,Н,К,win,-1,-1,none,-1,-1,none,Н
1836,289,6,Громовой Дубль,<=5,1,1.00,50,Н,Н,draw,...,К,Б,win,Н,К,win,-1,-1,none,Н
1837,289,7,Громовой Дубль,<=5,1,1.00,50,Н,Н,draw,...,К,Н,lose,К,Б,win,Н,К,win,Б


In [31]:
X, y = df_ml.drop('next_opp_move', axis=1), df_ml['next_opp_move']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [32]:
drop_features = ["player_name", "match_id"]
cat_features = [ "win_category", "opp_move", "my_move", "outcome", "prev1_opp_move", "prev1_my_move", "prev1_outcome", "prev2_opp_move", "prev2_my_move", "prev2_outcome"]
num_features = ["round", "stake", "opp_match_wins", "opp_match_winrate", "score_me_before", "score_opp_before", "streak_draws", "is_last_round"]
imputer_scaler_encoder = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        ('num', Pipeline([
            ('scaler', RobustScaler())
        ]), num_features),
        ('low_cat', Pipeline([
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preprocessor = Pipeline(
    [
        ('imputer_scaler_encoder', imputer_scaler_encoder)
    ]
)

In [33]:
preprocessor.fit(X_train, y_train)
preprocessed_X_train = preprocessor.transform(X_train)
preprocessed_X_valid = preprocessor.transform(X_valid)

In [34]:
preprocessed_X_train.shape

(783, 53)

In [21]:
# Убедимся, что preprocessed_X_train и preprocessed_X_valid созданы через новый preprocessor
# (это уже должно быть сделано, если вы перезапустили ячейки)

cb = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=3,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    random_strength=1,
    eval_metric='Accuracy',
    early_stopping_rounds=20,
    verbose=50,
    random_state=42
)

cb.fit(
    preprocessed_X_train, y_train,
    eval_set=(preprocessed_X_valid, y_valid),
    plot=True
)

NameError: name 'preprocessed_X_train' is not defined

In [38]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- 1. Загрузка данных и обученных объектов ----------
# Предполагается, что df, preprocessor, cb, prob_r2, prob_r3 уже существуют.
# Также нужны df_r2, df_r3 для fallback (если prob_r2/prob_r3 возвращают None).
# Убедитесь, что у вас есть эти объекты.

# ---------- 2. Вспомогательные функции ----------
def counter_move(move):
    """Возвращает контр-ход для заданного хода соперника"""
    if move == 'К': return 'Б'
    if move == 'Н': return 'К'
    if move == 'Б': return 'Н'
    return 'К'  # fallback

def get_move_ml(row_features):
    """Предсказание через ML модель (для раунда >=4)"""
    if isinstance(row_features, dict):
        X = pd.DataFrame([row_features])
    else:
        X = row_features
    X_processed = preprocessor.transform(X)
    pred = cb.predict(X_processed)
    # Извлекаем скаляр
    if isinstance(pred, np.ndarray):
        pred_code = pred.item() if pred.size == 1 else pred[0]
    else:
        pred_code = pred
    # Если pred_code уже строка ('К','Н','Б'), используем напрямую
    if pred_code in ('К', 'Н', 'Б'):
        predicted_opp = pred_code
    else:
        # иначе считаем, что это число 0,1,2
        move_map = {0: 'К', 1: 'Н', 2: 'Б'}
        predicted_opp = move_map[pred_code]
    return counter_move(predicted_opp)

# ---------- 3. Симуляция одного матча ----------
def simulate_match(match_df):
    """
    match_df : DataFrame с одним match_id, отсортированный по round.
    Возвращает: итоговый счёт (score_me - score_opp) и словарь статистики.
    """
    score_me = 0
    score_opp = 0
    n_rounds = len(match_df)
    # Проходим по раундам, но предсказываем ход для следующего раунда
    for i in range(n_rounds):
        curr = match_df.iloc[i]
        rnd = curr['round']
        # Для последнего раунда нет следующего хода — пропускаем
        if i == n_rounds - 1:
            break

        # Получаем реальный ход соперника в следующем раунде
        real_next_opp = match_df.iloc[i+1]['opp_move']

        # Выбираем стратегию в зависимости от номера раунда
        if rnd == 1:
            my_move = get_move_r1(curr['stake'])
        elif rnd == 2:
            # Данные первого раунда
            r1 = match_df.iloc[i-1]
            my_move = get_move_r2(curr['stake'],
                                  r1['outcome'], r1['my_move'], r1['opp_move'],
                                  prob_r2)
            if my_move is None:
                # fallback: использовать get_move_r1
                my_move = get_move_r1(curr['stake'])
        elif rnd == 3:
            # Данные первого и второго раундов
            r1 = match_df.iloc[i-2]
            r2 = match_df.iloc[i-1]
            my_move = get_move_r3(curr['stake'],
                                  r2['outcome'], r2['my_move'], r2['opp_move'],
                                  r1['outcome'], r1['my_move'], r1['opp_move'],
                                  prob_r3)
            if my_move is None:
                # fallback: использовать get_move_r2
                my_move = get_move_r2(curr['stake'],
                                      r2['outcome'], r2['my_move'], r2['opp_move'],
                                      prob_r2)
                if my_move is None:
                    my_move = get_move_r1(curr['stake'])
        else:  # rnd >= 4
            # Признаки для ML берутся из текущего раунда (который уже завершён)
            features = {
                'round': curr['round'],
                'stake': curr['stake'],
                'opp_match_wins': curr['opp_match_wins'],
                'opp_match_winrate': curr['opp_match_winrate'],
                'score_me_before': curr['score_me_before'],
                'score_opp_before': curr['score_opp_before'],
                'streak_draws': curr['streak_draws'],
                'is_last_round': curr['is_last_round'],
                'win_category': curr['win_category'],
                'opp_move': curr['opp_move'],
                'my_move': curr['my_move'],
                'outcome': curr['outcome'],
                'prev1_opp_move': curr['prev1_opp_move'],
                'prev1_my_move': curr['prev1_my_move'],
                'prev1_outcome': curr['prev1_outcome'],
                'prev2_opp_move': curr['prev2_opp_move'],
                'prev2_my_move': curr['prev2_my_move'],
                'prev2_outcome': curr['prev2_outcome']
            }
            my_move = get_move_ml(features)

        # Определяем исход раунда (мой ход vs реальный ход соперника)
        if my_move == real_next_opp:
            outcome = 'draw'
            delta_me, delta_opp = 0, 0
        elif (my_move == 'К' and real_next_opp == 'Н') or \
             (my_move == 'Н' and real_next_opp == 'Б') or \
             (my_move == 'Б' and real_next_opp == 'К'):
            outcome = 'win'
            delta_me, delta_opp = 1, 0
        else:
            outcome = 'loss'
            delta_me, delta_opp = 0, 1

        score_me += delta_me
        score_opp += delta_opp

    return score_me - score_opp, {'score_me': score_me, 'score_opp': score_opp}

# ---------- 4. Прогон по всем матчам ----------
results = []
for match_id, group in df.groupby('match_id'):
    group = group.sort_values('round')
    diff, stats = simulate_match(group)
    results.append({
        'match_id': match_id,
        'diff': diff,
        'score_me': stats['score_me'],
        'score_opp': stats['score_opp']
    })

results_df = pd.DataFrame(results)
print("=== Результаты стратегии (статистика + ML) ===")
print(f"Всего матчей: {len(results_df)}")
print(f"Средняя разница очков: {results_df['diff'].mean():.3f}")
print(f"Общий счёт (me - opp): {results_df['diff'].sum()}")
print(f"Победы (diff>0): {(results_df['diff']>0).sum()}")
print(f"Поражения (diff<0): {(results_df['diff']<0).sum()}")
print(f"Ничьи (diff==0): {(results_df['diff']==0).sum()}")

# ---------- 5. Бейзлайн: всегда камень ----------
def always_rock_simulation(match_df):
    score_me = 0
    score_opp = 0
    for i in range(len(match_df)-1):
        real_next_opp = match_df.iloc[i+1]['opp_move']
        my_move = 'К'
        if my_move == real_next_opp:
            continue
        elif (my_move == 'К' and real_next_opp == 'Н') or \
             (my_move == 'Н' and real_next_opp == 'Б') or \
             (my_move == 'Б' and real_next_opp == 'К'):
            score_me += 1
        else:
            score_opp += 1
    return score_me - score_opp

baseline_diffs = []
for match_id, group in df.groupby('match_id'):
    group = group.sort_values('round')
    diff = always_rock_simulation(group)
    baseline_diffs.append(diff)

print("\n=== Бейзлайн: всегда камень ===")
print(f"Средняя разница очков: {np.mean(baseline_diffs):.3f}")
print(f"Общий счёт: {np.sum(baseline_diffs)}")

=== Результаты стратегии (статистика + ML) ===
Всего матчей: 288
Средняя разница очков: 0.267
Общий счёт (me - opp): 77
Победы (diff>0): 122
Поражения (diff<0): 99
Ничьи (diff==0): 67

=== Бейзлайн: всегда камень ===
Средняя разница очков: -0.125
Общий счёт: -36


In [47]:
import joblib
joblib.dump(cb, 'rps_model.pkl')

['rps_model.pkl']

In [ ]:
# df.to_csv('/content/drive/MyDrive/rps_data.csv', sep=',', index=False)